# AAI 501 Final Team Project

## Intelligent Predictive Maintenance for Turbofan Engines Using Remaining Useful Life Prediction

**Team 6:** Sudhakaran Srinivasan, Mina Habib, Ivan Da Silva, Russell Miller

**Dataset:** NASA C-MAPSS — FD001  
**Primary target:** Remaining Useful Life (RUL)

### End-to-end workflow
1. Data preparation and RUL construction
2. EDA and degradation analysis
3. Baseline machine-learning models
4. XGBoost and explainability
5. LSTM sequence modeling
6. Integrated model comparison
7. PEAS-based intelligent maintenance decision agent
8. Capacity-constrained maintenance queue and policy evaluation

Run the notebook from top to bottom. Place `CMAPSSData.zip` in the project root,
`data/raw/`, or `/mnt/data/CMAPSSData.zip`.

## Executive Summary

This project predicts Remaining Useful Life from multivariate turbofan-engine sensor data
and converts the selected model's predictions into a capacity-constrained maintenance queue.
Its differentiator is a classical utility-based decision agent framed using the PEAS methodology.

# Part 1 — Data Preparation, EDA, and Degradation Analysis

In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42
RUL_CAP = 125

## 1. Project paths and dataset extraction

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "eda"

for folder in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

candidate_zips = [
    PROJECT_ROOT / "CMAPSSData.zip",
    RAW_DIR / "CMAPSSData.zip",
    Path("/mnt/data/CMAPSSData.zip"),
]

zip_path = next((p for p in candidate_zips if p.exists()), None)
if zip_path is None:
    raise FileNotFoundError(
        "CMAPSSData.zip was not found. Place it in the project root or data/raw/."
    )

with zipfile.ZipFile(zip_path, "r") as zf:
    needed = ["train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt", "readme.txt"]
    for name in needed:
        target = RAW_DIR / name
        if not target.exists():
            zf.extract(name, RAW_DIR)

print(f"Using dataset: {zip_path}")
print("Extracted files:", sorted(p.name for p in RAW_DIR.glob("*FD001*")))

## 2. Load FD001 and assign column names

In [ ]:
columns = (
    ["unit_id", "cycle", "setting_1", "setting_2", "setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train = pd.read_csv(
    RAW_DIR / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

test = pd.read_csv(
    RAW_DIR / "test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

test_rul = pd.read_csv(
    RAW_DIR / "RUL_FD001.txt",
    sep=r"\s+",
    header=None,
    names=["rul_at_last_observation"]
)

test_rul["unit_id"] = np.arange(1, len(test_rul) + 1)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Test RUL shape:", test_rul.shape)
display(train.head())

## 3. Data-quality checks

In [ ]:
quality_summary = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing_train": train.isna().sum(),
    "missing_test": test.isna().sum(),
    "nunique_train": train.nunique(),
    "variance_train": train.var(numeric_only=True)
})

print("Duplicate training rows:", train.duplicated().sum())
print("Duplicate test rows:", test.duplicated().sum())
print("Training engines:", train["unit_id"].nunique())
print("Test engines:", test["unit_id"].nunique())
display(quality_summary)

## 4. Construct Remaining Useful Life

For each training row:

\[
RUL = 	ext{maximum cycle for that engine} - 	ext{current cycle}
\]

We retain both the uncapped and capped targets. The capped target treats the early
healthy period as a plateau and is commonly used in C-MAPSS modeling.

In [ ]:
max_cycles = (
    train.groupby("unit_id")["cycle"]
    .max()
    .rename("max_cycle")
    .reset_index()
)

train = train.merge(max_cycles, on="unit_id", how="left")
train["rul"] = train["max_cycle"] - train["cycle"]
train["rul_capped"] = train["rul"].clip(upper=RUL_CAP)

# Ground truth for each test engine at its final observed cycle
test_last = (
    test.sort_values(["unit_id", "cycle"])
    .groupby("unit_id", as_index=False)
    .tail(1)
    .merge(test_rul, on="unit_id", how="left")
    .rename(columns={"rul_at_last_observation": "actual_rul"})
    .reset_index(drop=True)
)

assert train["rul"].min() == 0
assert test_last["actual_rul"].notna().all()

display(train[["unit_id", "cycle", "max_cycle", "rul", "rul_capped"]].head())
display(test_last[["unit_id", "cycle", "actual_rul"]].head())

## 5. Engine-lifetime analysis

In [ ]:
engine_life = train.groupby("unit_id")["cycle"].max()

print(engine_life.describe())

plt.figure(figsize=(8, 5))
plt.hist(engine_life, bins=15, edgecolor="black")
plt.title("Distribution of Training Engine Lifetimes")
plt.xlabel("Maximum Operational Cycle")
plt.ylabel("Number of Engines")
plt.tight_layout()
plt.show()

## 6. Identify low-variance sensors

In [ ]:
sensor_cols = [f"sensor_{i}" for i in range(1, 22)]
setting_cols = ["setting_1", "setting_2", "setting_3"]

sensor_variance = train[sensor_cols].var().sort_values()
display(sensor_variance.to_frame("variance"))

VARIANCE_THRESHOLD = 1e-8
low_variance_sensors = sensor_variance[
    sensor_variance <= VARIANCE_THRESHOLD
].index.tolist()

selected_sensors = [s for s in sensor_cols if s not in low_variance_sensors]

print("Low-variance sensors removed:", low_variance_sensors)
print("Selected sensors:", selected_sensors)

## 7. Correlation with cycle and RUL

In [ ]:
correlation_table = pd.DataFrame({
    "corr_with_cycle": train[selected_sensors].corrwith(train["cycle"]),
    "corr_with_rul": train[selected_sensors].corrwith(train["rul_capped"])
}).sort_values("corr_with_rul", key=lambda s: s.abs(), ascending=False)

display(correlation_table)

## 8. Plot representative degradation trajectories

In [ ]:
top_sensors = correlation_table.head(6).index.tolist()
example_engines = [1, 20, 50, 75, 100]

for sensor in top_sensors[:4]:
    plt.figure(figsize=(9, 5))
    for engine_id in example_engines:
        engine = train[train["unit_id"] == engine_id]
        life_fraction = engine["cycle"] / engine["max_cycle"]
        plt.plot(life_fraction, engine[sensor], label=f"Engine {engine_id}", alpha=0.8)

    plt.title(f"{sensor}: Sensor Trajectory Across Normalized Engine Life")
    plt.xlabel("Fraction of Engine Life Completed")
    plt.ylabel(sensor)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 9. Compare early-, middle-, and late-life sensor behavior

In [ ]:
train["life_stage"] = pd.cut(
    train["rul"],
    bins=[-1, 30, 100, np.inf],
    labels=["Late life (RUL ≤ 30)", "Middle life", "Early life"]
)

stage_summary = (
    train.groupby("life_stage", observed=False)[top_sensors]
    .mean()
    .T
)
display(stage_summary)

for sensor in top_sensors[:4]:
    data = [
        train.loc[train["life_stage"] == stage, sensor].dropna()
        for stage in ["Early life", "Middle life", "Late life (RUL ≤ 30)"]
    ]
    plt.figure(figsize=(8, 5))
    plt.boxplot(data, tick_labels=["Early", "Middle", "Late"], showfliers=False)
    plt.title(f"{sensor} by Engine Life Stage")
    plt.xlabel("Life Stage")
    plt.ylabel(sensor)
    plt.tight_layout()
    plt.show()

## 10. Create leakage-safe rolling features

Rolling features are calculated separately within each engine and use only the
current and previous cycles. This avoids using future observations.

In [ ]:
def add_rolling_features(df, sensors, windows=(5, 10)):
    result = df.sort_values(["unit_id", "cycle"]).copy()
    grouped = result.groupby("unit_id", group_keys=False)

    for sensor in sensors:
        result[f"{sensor}_diff_1"] = grouped[sensor].diff().fillna(0)

        for window in windows:
            result[f"{sensor}_roll_mean_{window}"] = (
                grouped[sensor]
                .rolling(window, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )
            result[f"{sensor}_roll_std_{window}"] = (
                grouped[sensor]
                .rolling(window, min_periods=2)
                .std()
                .reset_index(level=0, drop=True)
                .fillna(0)
            )

    return result

# Limit engineered features to the six most RUL-correlated sensors
engineered_sensors = top_sensors[:6]

train_features = add_rolling_features(train, engineered_sensors)
test_features = add_rolling_features(test, engineered_sensors)

print("Engineered sensors:", engineered_sensors)
print("Train feature shape:", train_features.shape)
print("Test feature shape:", test_features.shape)

## 11. Save processed datasets and metadata

In [ ]:
train_features.to_csv(PROCESSED_DIR / "fd001_train_processed.csv", index=False)
test_features.to_csv(PROCESSED_DIR / "fd001_test_processed.csv", index=False)
test_last.to_csv(PROCESSED_DIR / "fd001_test_last_with_rul.csv", index=False)

metadata = {
    "rul_cap": RUL_CAP,
    "selected_sensors": selected_sensors,
    "engineered_sensors": engineered_sensors,
    "low_variance_sensors": low_variance_sensors,
    "train_rows": int(len(train_features)),
    "test_rows": int(len(test_features)),
    "train_engines": int(train_features["unit_id"].nunique()),
    "test_engines": int(test_features["unit_id"].nunique())
}

import json
with open(PROCESSED_DIR / "fd001_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved processed files to:", PROCESSED_DIR)
print(json.dumps(metadata, indent=2))

## Notebook 01 conclusions

Complete this Markdown cell after running the notebook:

- FD001 contains ___ training rows and ___ test rows.
- The training set contains ___ engines.
- The most informative sensors by absolute RUL correlation were ___.
- The following sensors were removed because they were nearly constant: ___.
- The degradation plots indicate that ___.

# Part 2 — Baseline Machine-Learning Models

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PREDICTIONS_DIR = PROJECT_ROOT / "data" / "predictions"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "baseline_models"

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(PROCESSED_DIR / "fd001_train_processed.csv")
test = pd.read_csv(PROCESSED_DIR / "fd001_test_processed.csv")
test_last_truth = pd.read_csv(PROCESSED_DIR / "fd001_test_last_with_rul.csv")

with open(PROCESSED_DIR / "fd001_metadata.json") as f:
    metadata = json.load(f)

display(train.head())
print(metadata)

## 1. Define evaluation metrics

In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

def cmapss_score(y_true, y_pred):
    error = np.asarray(y_pred) - np.asarray(y_true)
    penalties = np.where(
        error < 0,
        np.exp(-error / 13) - 1,
        np.exp(error / 10) - 1
    )
    return float(penalties.sum())

def evaluate_regression(y_true, y_pred, model_name):
    return {
        "model": model_name,
        "rmse": rmse(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "cmapss_score": cmapss_score(y_true, y_pred)
    }

## 2. Select features and target

In [ ]:
excluded = {
    "unit_id", "max_cycle", "rul", "rul_capped", "life_stage"
}

feature_cols = [
    c for c in train.columns
    if c not in excluded
]

target_col = "rul_capped"

print("Number of features:", len(feature_cols))
print(feature_cols[:20])

## 3. Group-based train/validation split

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, val_idx = next(
    splitter.split(
        train[feature_cols],
        train[target_col],
        groups=train["unit_id"]
    )
)

train_part = train.iloc[train_idx].copy()
val_part = train.iloc[val_idx].copy()

X_train = train_part[feature_cols]
y_train = train_part[target_col]
X_val = val_part[feature_cols]
y_val = val_part[target_col]

print("Training engines:", train_part["unit_id"].nunique())
print("Validation engines:", val_part["unit_id"].nunique())
print("Engine overlap:", set(train_part["unit_id"]) & set(val_part["unit_id"]))

## 4. Mean-prediction baseline

In [ ]:
mean_prediction = np.repeat(y_train.mean(), len(y_val))
baseline_result = evaluate_regression(y_val, mean_prediction, "Mean Baseline")
baseline_result

## 5. Ridge Regression

In [ ]:
ridge_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10.0))
])

ridge_pipeline.fit(X_train, y_train)
ridge_val_pred = ridge_pipeline.predict(X_val)
ridge_result = evaluate_regression(y_val, ridge_val_pred, "Ridge Regression")
ridge_result

## 6. Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=3,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_model.fit(X_train, y_train)
rf_val_pred = rf_model.predict(X_val)
rf_result = evaluate_regression(y_val, rf_val_pred, "Random Forest")
rf_result

## 7. Compare validation performance

In [ ]:
validation_results = pd.DataFrame([
    baseline_result,
    ridge_result,
    rf_result
]).sort_values("rmse")

display(validation_results)
validation_results.to_csv(
    OUTPUT_DIR / "baseline_validation_results.csv",
    index=False
)

## 8. Residual analysis

In [ ]:
residuals = y_val.to_numpy() - rf_val_pred

plt.figure(figsize=(8, 5))
plt.scatter(rf_val_pred, residuals, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.title("Random Forest Validation Residuals")
plt.xlabel("Predicted Capped RUL")
plt.ylabel("Actual - Predicted")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=30, edgecolor="black")
plt.title("Random Forest Residual Distribution")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 9. Feature importance

In [ ]:
rf_importance = (
    pd.DataFrame({
        "feature": feature_cols,
        "importance": rf_model.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .head(20)
)

display(rf_importance)

plt.figure(figsize=(9, 7))
plt.barh(
    rf_importance["feature"][::-1],
    rf_importance["importance"][::-1]
)
plt.title("Random Forest: Top 20 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 10. Retrain on all training engines and evaluate the final test observation

C-MAPSS test ground truth is provided for each engine's last observed cycle.
Therefore, test evaluation uses one row per engine.

In [ ]:
test_last_features = (
    test.sort_values(["unit_id", "cycle"])
    .groupby("unit_id", as_index=False)
    .tail(1)
    .sort_values("unit_id")
    .reset_index(drop=True)
)

test_eval = test_last_features.merge(
    test_last_truth[["unit_id", "actual_rul"]],
    on="unit_id",
    how="left"
)

X_all = train[feature_cols]
y_all = train[target_col]
X_test_last = test_eval[feature_cols]
y_test = test_eval["actual_rul"]

ridge_pipeline.fit(X_all, y_all)
rf_model.fit(X_all, y_all)

ridge_test_pred = np.clip(ridge_pipeline.predict(X_test_last), 0, None)
rf_test_pred = np.clip(rf_model.predict(X_test_last), 0, None)

test_results = pd.DataFrame([
    evaluate_regression(y_test, ridge_test_pred, "Ridge Regression"),
    evaluate_regression(y_test, rf_test_pred, "Random Forest")
]).sort_values("rmse")

display(test_results)
test_results.to_csv(OUTPUT_DIR / "baseline_test_results.csv", index=False)

## 11. Save model predictions

In [ ]:
ridge_predictions = pd.DataFrame({
    "unit_id": test_eval["unit_id"],
    "actual_rul": y_test,
    "predicted_rul": ridge_test_pred,
    "model": "Ridge Regression"
})

rf_predictions = pd.DataFrame({
    "unit_id": test_eval["unit_id"],
    "actual_rul": y_test,
    "predicted_rul": rf_test_pred,
    "model": "Random Forest"
})

ridge_predictions.to_csv(
    PREDICTIONS_DIR / "ridge_test_predictions.csv",
    index=False
)
rf_predictions.to_csv(
    PREDICTIONS_DIR / "random_forest_test_predictions.csv",
    index=False
)

display(rf_predictions.head())

## Notebook 02 conclusions

Complete after execution:

- The mean baseline RMSE was ___.
- Ridge Regression achieved ___ RMSE.
- Random Forest achieved ___ RMSE.
- The strongest baseline model was ___.
- The most influential features were ___.

# Part 3 — Advanced ML: XGBoost and Explainability

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit, RandomizedSearchCV, GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

RANDOM_STATE = 42

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PREDICTIONS_DIR = PROJECT_ROOT / "data" / "predictions"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "advanced_ml"

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(PROCESSED_DIR / "fd001_train_processed.csv")
test = pd.read_csv(PROCESSED_DIR / "fd001_test_processed.csv")
test_last_truth = pd.read_csv(PROCESSED_DIR / "fd001_test_last_with_rul.csv")

excluded = {"unit_id", "max_cycle", "rul", "rul_capped", "life_stage"}
feature_cols = [c for c in train.columns if c not in excluded]
target_col = "rul_capped"

print("Features:", len(feature_cols))

## 1. Metrics

In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

def cmapss_score(y_true, y_pred):
    error = np.asarray(y_pred) - np.asarray(y_true)
    penalty = np.where(
        error < 0,
        np.exp(-error / 13) - 1,
        np.exp(error / 10) - 1
    )
    return float(penalty.sum())

def evaluate(y_true, y_pred, model_name):
    return {
        "model": model_name,
        "rmse": rmse(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "cmapss_score": cmapss_score(y_true, y_pred)
    }

## 2. Engine-level split

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, val_idx = next(
    splitter.split(
        train[feature_cols],
        train[target_col],
        groups=train["unit_id"]
    )
)

train_part = train.iloc[train_idx]
val_part = train.iloc[val_idx]

X_train = train_part[feature_cols]
y_train = train_part[target_col]
X_val = val_part[feature_cols]
y_val = val_part[target_col]

print("Training engines:", train_part["unit_id"].nunique())
print("Validation engines:", val_part["unit_id"].nunique())

## 3. Initial XGBoost model

In [ ]:
xgb_initial = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=600,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

xgb_initial.fit(X_train, y_train)
initial_pred = np.clip(xgb_initial.predict(X_val), 0, None)
initial_result = evaluate(y_val, initial_pred, "Initial XGBoost")
initial_result

## 4. Modest hyperparameter search

The search is intentionally limited so the project remains feasible. GroupKFold
keeps all rows from a given engine in the same fold.

In [ ]:
param_distributions = {
    "n_estimators": [300, 500, 700],
    "learning_rate": [0.02, 0.04, 0.06],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.75, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_alpha": [0.0, 0.05, 0.2],
    "reg_lambda": [0.8, 1.0, 2.0]
}

base_xgb = XGBRegressor(
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

group_cv = GroupKFold(n_splits=3)

search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="neg_root_mean_squared_error",
    cv=group_cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

search.fit(
    X_train,
    y_train,
    groups=train_part["unit_id"]
)

print("Best parameters:")
print(search.best_params_)
print("Best CV RMSE:", -search.best_score_)

## 5. Validation evaluation

In [ ]:
best_xgb = search.best_estimator_
tuned_pred = np.clip(best_xgb.predict(X_val), 0, None)

tuned_result = evaluate(y_val, tuned_pred, "Tuned XGBoost")
comparison = pd.DataFrame([initial_result, tuned_result]).sort_values("rmse")
display(comparison)
comparison.to_csv(OUTPUT_DIR / "xgboost_validation_results.csv", index=False)

## 6. Residual analysis

In [ ]:
residuals = y_val.to_numpy() - tuned_pred

plt.figure(figsize=(8, 5))
plt.scatter(tuned_pred, residuals, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.title("Tuned XGBoost Validation Residuals")
plt.xlabel("Predicted Capped RUL")
plt.ylabel("Actual - Predicted")
plt.tight_layout()
plt.show()

## 7. Built-in feature importance

In [ ]:
importance = (
    pd.DataFrame({
        "feature": feature_cols,
        "importance": best_xgb.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .head(20)
)

display(importance)

plt.figure(figsize=(9, 7))
plt.barh(importance["feature"][::-1], importance["importance"][::-1])
plt.title("XGBoost: Top 20 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 8. SHAP analysis

SHAP is optional. If it is unavailable, the notebook will continue and the
built-in feature-importance analysis remains valid.

In [ ]:
try:
    import shap

    sample_size = min(1500, len(X_val))
    X_shap = X_val.sample(sample_size, random_state=RANDOM_STATE)

    explainer = shap.TreeExplainer(best_xgb)
    shap_values = explainer.shap_values(X_shap)

    shap.summary_plot(
        shap_values,
        X_shap,
        max_display=15,
        show=False
    )
    plt.tight_layout()
    plt.show()

except Exception as exc:
    print("SHAP analysis skipped:", exc)

## 9. Retrain on all training engines and evaluate test engines

In [ ]:
test_last_features = (
    test.sort_values(["unit_id", "cycle"])
    .groupby("unit_id", as_index=False)
    .tail(1)
    .sort_values("unit_id")
    .reset_index(drop=True)
)

test_eval = test_last_features.merge(
    test_last_truth[["unit_id", "actual_rul"]],
    on="unit_id",
    how="left"
)

final_xgb = XGBRegressor(
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    **search.best_params_
)

final_xgb.fit(train[feature_cols], train[target_col])

xgb_test_pred = np.clip(
    final_xgb.predict(test_eval[feature_cols]),
    0,
    None
)

test_result = evaluate(
    test_eval["actual_rul"],
    xgb_test_pred,
    "Tuned XGBoost"
)

display(pd.DataFrame([test_result]))

## 10. Save test predictions

In [ ]:
xgb_predictions = pd.DataFrame({
    "unit_id": test_eval["unit_id"],
    "actual_rul": test_eval["actual_rul"],
    "predicted_rul": xgb_test_pred,
    "model": "Tuned XGBoost"
})

xgb_predictions.to_csv(
    PREDICTIONS_DIR / "xgboost_test_predictions.csv",
    index=False
)

pd.DataFrame([test_result]).to_csv(
    OUTPUT_DIR / "xgboost_test_results.csv",
    index=False
)

display(xgb_predictions.head())

## Notebook 03 conclusions

Complete after execution:

- Tuned XGBoost achieved validation RMSE of ___.
- Its FD001 test RMSE was ___.
- The most influential features were ___.
- SHAP showed that higher/lower values of ___ tended to increase predicted RUL.
- Compared with Random Forest, XGBoost performed ___.

# Part 4 — Deep Learning: LSTM Sequence Model

In [ ]:
from pathlib import Path
import json
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

RANDOM_STATE = 42
SEQUENCE_LENGTH = 30
BATCH_SIZE = 256
MAX_EPOCHS = 60
PATIENCE = 8

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", DEVICE)

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PREDICTIONS_DIR = PROJECT_ROOT / "data" / "predictions"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "deep_learning"

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(PROCESSED_DIR / "fd001_train_processed.csv")
test = pd.read_csv(PROCESSED_DIR / "fd001_test_processed.csv")
test_last_truth = pd.read_csv(PROCESSED_DIR / "fd001_test_last_with_rul.csv")

with open(PROCESSED_DIR / "fd001_metadata.json") as f:
    metadata = json.load(f)

# Use raw operating settings and selected nonconstant sensors.
sequence_features = (
    ["setting_1", "setting_2", "setting_3"]
    + metadata["selected_sensors"]
)

print("Sequence features:", sequence_features)

## 1. Engine-level split

In [ ]:
engine_ids = train["unit_id"].drop_duplicates().to_numpy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_engine_idx, val_engine_idx = next(
    splitter.split(
        engine_ids,
        groups=engine_ids
    )
)

train_engines = set(engine_ids[train_engine_idx])
val_engines = set(engine_ids[val_engine_idx])

train_part = train[train["unit_id"].isin(train_engines)].copy()
val_part = train[train["unit_id"].isin(val_engines)].copy()

print("Training engines:", len(train_engines))
print("Validation engines:", len(val_engines))
print("Overlap:", train_engines & val_engines)

## 2. Standardize sequence features

In [ ]:
scaler = StandardScaler()
scaler.fit(train_part[sequence_features])

train_part.loc[:, sequence_features] = scaler.transform(
    train_part[sequence_features]
)
val_part.loc[:, sequence_features] = scaler.transform(
    val_part[sequence_features]
)

test_scaled = test.copy()
test_scaled.loc[:, sequence_features] = scaler.transform(
    test_scaled[sequence_features]
)

## 3. Create sequence windows

In [ ]:
def build_train_sequences(df, features, target, sequence_length):
    X, y, unit_ids = [], [], []

    for unit_id, engine in df.groupby("unit_id"):
        engine = engine.sort_values("cycle")
        values = engine[features].to_numpy(dtype=np.float32)
        targets = engine[target].to_numpy(dtype=np.float32)

        if len(engine) < sequence_length:
            continue

        for end in range(sequence_length, len(engine) + 1):
            start = end - sequence_length
            X.append(values[start:end])
            y.append(targets[end - 1])
            unit_ids.append(unit_id)

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        np.asarray(unit_ids)
    )

X_train_seq, y_train_seq, train_seq_units = build_train_sequences(
    train_part,
    sequence_features,
    "rul_capped",
    SEQUENCE_LENGTH
)

X_val_seq, y_val_seq, val_seq_units = build_train_sequences(
    val_part,
    sequence_features,
    "rul_capped",
    SEQUENCE_LENGTH
)

print("Training sequence shape:", X_train_seq.shape)
print("Validation sequence shape:", X_val_seq.shape)

## 4. PyTorch dataset and loaders

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(
    SequenceDataset(X_train_seq, y_train_seq),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    SequenceDataset(X_val_seq, y_val_seq),
    batch_size=BATCH_SIZE,
    shuffle=False
)

## 5. Define the LSTM model

In [ ]:
class RULLSTM(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size=64,
        num_layers=1,
        dropout=0.20
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.regressor = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        output, _ = self.lstm(x)
        last_hidden = output[:, -1, :]
        return self.regressor(last_hidden)

model = RULLSTM(
    input_size=len(sequence_features),
    hidden_size=64,
    num_layers=1,
    dropout=0.20
).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

print(model)

## 6. Train with early stopping

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)

    total_loss = 0.0
    total_examples = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if is_training:
            optimizer.zero_grad()

        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        if is_training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples

history = {"train_loss": [], "val_loss": []}
best_state = None
best_val_loss = np.inf
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = run_epoch(
        model, train_loader, criterion, optimizer
    )
    val_loss = run_epoch(
        model, val_loader, criterion
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(
        f"Epoch {epoch:02d} | "
        f"Train RMSE: {train_loss ** 0.5:.3f} | "
        f"Val RMSE: {val_loss ** 0.5:.3f}"
    )

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

model.load_state_dict(best_state)
torch.save(model.state_dict(), OUTPUT_DIR / "best_lstm_state.pt")

## 7. Plot training history

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(np.sqrt(history["train_loss"]), label="Train RMSE")
plt.plot(np.sqrt(history["val_loss"]), label="Validation RMSE")
plt.title("LSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Validation metrics

In [ ]:
def predict_loader(model, loader):
    model.eval()
    outputs = []

    with torch.no_grad():
        for X_batch, _ in loader:
            predictions = model(X_batch.to(DEVICE))
            outputs.append(predictions.cpu().numpy().ravel())

    return np.concatenate(outputs)

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

def cmapss_score(y_true, y_pred):
    error = np.asarray(y_pred) - np.asarray(y_true)
    penalty = np.where(
        error < 0,
        np.exp(-error / 13) - 1,
        np.exp(error / 10) - 1
    )
    return float(penalty.sum())

val_pred = np.clip(predict_loader(model, val_loader), 0, None)

validation_metrics = {
    "model": "LSTM",
    "rmse": rmse(y_val_seq, val_pred),
    "mae": mean_absolute_error(y_val_seq, val_pred),
    "cmapss_score": cmapss_score(y_val_seq, val_pred)
}

validation_metrics

## 9. Build one final sequence per test engine

In [ ]:
def build_test_sequences(df, features, sequence_length):
    sequences = []
    unit_ids = []

    for unit_id, engine in df.groupby("unit_id"):
        engine = engine.sort_values("cycle")
        values = engine[features].to_numpy(dtype=np.float32)

        if len(values) >= sequence_length:
            sequence = values[-sequence_length:]
        else:
            pad_count = sequence_length - len(values)
            padding = np.repeat(
                values[[0]],
                repeats=pad_count,
                axis=0
            )
            sequence = np.vstack([padding, values])

        sequences.append(sequence)
        unit_ids.append(unit_id)

    return (
        np.asarray(sequences, dtype=np.float32),
        np.asarray(unit_ids)
    )

X_test_seq, test_unit_ids = build_test_sequences(
    test_scaled,
    sequence_features,
    SEQUENCE_LENGTH
)

dummy_y = np.zeros(len(X_test_seq), dtype=np.float32)
test_loader = DataLoader(
    SequenceDataset(X_test_seq, dummy_y),
    batch_size=BATCH_SIZE,
    shuffle=False
)

lstm_test_pred = np.clip(
    predict_loader(model, test_loader),
    0,
    None
)

print("Test sequence shape:", X_test_seq.shape)

## 10. Evaluate against NASA test RUL

In [ ]:
test_truth = (
    test_last_truth[["unit_id", "actual_rul"]]
    .sort_values("unit_id")
    .reset_index(drop=True)
)

assert np.array_equal(test_unit_ids, test_truth["unit_id"].to_numpy())

y_test = test_truth["actual_rul"].to_numpy()

test_metrics = {
    "model": "LSTM",
    "rmse": rmse(y_test, lstm_test_pred),
    "mae": mean_absolute_error(y_test, lstm_test_pred),
    "cmapss_score": cmapss_score(y_test, lstm_test_pred)
}

display(pd.DataFrame([validation_metrics]))
display(pd.DataFrame([test_metrics]))

## 11. Save predictions

In [ ]:
lstm_predictions = pd.DataFrame({
    "unit_id": test_unit_ids,
    "actual_rul": y_test,
    "predicted_rul": lstm_test_pred,
    "model": "LSTM"
})

lstm_predictions.to_csv(
    PREDICTIONS_DIR / "lstm_test_predictions.csv",
    index=False
)

pd.DataFrame([validation_metrics]).to_csv(
    OUTPUT_DIR / "lstm_validation_results.csv",
    index=False
)

pd.DataFrame([test_metrics]).to_csv(
    OUTPUT_DIR / "lstm_test_results.csv",
    index=False
)

display(lstm_predictions.head())

## 12. Actual versus predicted RUL

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, lstm_test_pred, alpha=0.65)
diagonal_max = max(y_test.max(), lstm_test_pred.max())
plt.plot([0, diagonal_max], [0, diagonal_max], linestyle="--")
plt.title("LSTM: Actual vs. Predicted Test RUL")
plt.xlabel("Actual RUL")
plt.ylabel("Predicted RUL")
plt.tight_layout()
plt.show()

## Notebook 04 conclusions

Complete after execution:

- The LSTM used a sequence length of 30 cycles and ___ input features.
- Its validation RMSE was ___.
- Its FD001 test RMSE was ___.
- Compared with XGBoost, the LSTM performed ___.
- The result suggests that temporal sequence modeling ___.

# Part 5 — Integrated Model Comparison

This section collects all saved test predictions and selects the model used by the
maintenance agent. The ranking prioritizes the asymmetric C-MAPSS score, then RMSE and MAE.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PREDICTIONS_DIR = PROJECT_ROOT / "data" / "predictions"
FINAL_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "final_analysis"
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def final_rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def final_mae(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))

def final_cmapss_score(y_true, y_pred):
    error = np.asarray(y_pred) - np.asarray(y_true)
    penalties = np.where(
        error < 0,
        np.exp(-error / 13) - 1,
        np.exp(error / 10) - 1
    )
    return float(penalties.sum())

In [ ]:
prediction_files = {
    "Ridge Regression": PREDICTIONS_DIR / "ridge_test_predictions.csv",
    "Random Forest": PREDICTIONS_DIR / "random_forest_test_predictions.csv",
    "Tuned XGBoost": PREDICTIONS_DIR / "xgboost_test_predictions.csv",
    "LSTM": PREDICTIONS_DIR / "lstm_test_predictions.csv",
}

model_predictions = {}
comparison_rows = []

for model_name, path in prediction_files.items():
    if not path.exists():
        print(f"Skipping {model_name}: {path.name} was not found.")
        continue

    pred = pd.read_csv(path)
    required = {"unit_id", "actual_rul", "predicted_rul"}
    missing = required - set(pred.columns)
    if missing:
        raise ValueError(f"{model_name} is missing columns: {missing}")

    pred["predicted_rul"] = pred["predicted_rul"].clip(lower=0)
    model_predictions[model_name] = pred.copy()

    comparison_rows.append({
        "model": model_name,
        "rmse": final_rmse(pred["actual_rul"], pred["predicted_rul"]),
        "mae": final_mae(pred["actual_rul"], pred["predicted_rul"]),
        "cmapss_score": final_cmapss_score(
            pred["actual_rul"], pred["predicted_rul"]
        )
    })

if not comparison_rows:
    raise FileNotFoundError("Run Parts 2–4 first so prediction files are created.")

final_model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(["cmapss_score", "rmse", "mae"])
    .reset_index(drop=True)
)

display(final_model_comparison)
final_model_comparison.to_csv(
    FINAL_OUTPUT_DIR / "final_model_comparison.csv", index=False
)

best_model_name = final_model_comparison.iloc[0]["model"]
best_predictions = model_predictions[best_model_name].copy()
print("Selected model for the maintenance agent:", best_model_name)

In [ ]:
plt.figure(figsize=(9, 5))
order = final_model_comparison.sort_values("rmse")
plt.bar(order["model"], order["rmse"])
plt.title("FD001 Test RMSE by Model")
plt.xlabel("Model")
plt.ylabel("RMSE")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
order = final_model_comparison.sort_values("cmapss_score")
plt.bar(order["model"], order["cmapss_score"])
plt.title("Asymmetric C-MAPSS Score by Model")
plt.xlabel("Model")
plt.ylabel("C-MAPSS Score — Lower Is Better")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# Part 6 — Intelligent Maintenance Decision Agent

Prediction alone does not tell an airline which engine to service first.
The agent converts predicted RUL into an operational action while respecting limited capacity.

## PEAS Methodology

| PEAS component | Project definition |
|---|---|
| **Performance measure** | Minimize late-maintenance risk, maximize critical-engine selection, minimize queue regret, and respect capacity |
| **Environment** | A fleet of degrading turbofan engines and a maintenance facility with limited service slots |
| **Actuators** | Select an engine, assign queue position, or defer it |
| **Sensors / percepts** | Predicted RUL, derived risk tier, optional uncertainty, and maintenance capacity |

**Agent cycle:** Perceive → Estimate health → Evaluate urgency → Choose action → Generate queue → Evaluate outcome

## Agent Architecture

```text
Engine sensor history
        ↓
Best RUL prediction model
        ↓
Predicted RUL and risk tier
        ↓
Utility-based maintenance agent
        ↓
Capacity-constrained maintenance queue
```

In [ ]:
CRITICAL_THRESHOLD = 15
WARNING_THRESHOLD = 40

def assign_risk_tier(rul):
    if pd.isna(rul):
        return "Unknown"
    if rul <= CRITICAL_THRESHOLD:
        return "Critical"
    if rul <= WARNING_THRESHOLD:
        return "Warning"
    return "Healthy"

best_predictions["predicted_rul_raw"] = best_predictions["predicted_rul"]
best_predictions["predicted_rul"] = best_predictions["predicted_rul"].clip(lower=0)
best_predictions["error"] = (
    best_predictions["predicted_rul"] - best_predictions["actual_rul"]
)
best_predictions["absolute_error"] = best_predictions["error"].abs()
best_predictions["predicted_risk_tier"] = (
    best_predictions["predicted_rul"].apply(assign_risk_tier)
)
best_predictions["actual_risk_tier"] = (
    best_predictions["actual_rul"].apply(assign_risk_tier)
)

RISK_SEVERITY = {"Healthy": 0, "Warning": 1, "Critical": 2, "Unknown": 0}
best_predictions["predicted_risk_score"] = (
    best_predictions["predicted_risk_tier"].map(RISK_SEVERITY)
)

best_predictions["rul_urgency"] = 1 / (best_predictions["predicted_rul"] + 1)
u_min = best_predictions["rul_urgency"].min()
u_max = best_predictions["rul_urgency"].max()
best_predictions["rul_urgency_normalized"] = (
    (best_predictions["rul_urgency"] - u_min) / (u_max - u_min)
    if u_max > u_min else 0.0
)

best_predictions["risk_normalized"] = (
    best_predictions["predicted_risk_score"] / 2
)

if "prediction_uncertainty" not in best_predictions.columns:
    best_predictions["prediction_uncertainty"] = 0.0

q_min = best_predictions["prediction_uncertainty"].min()
q_max = best_predictions["prediction_uncertainty"].max()
best_predictions["uncertainty_normalized"] = (
    (best_predictions["prediction_uncertainty"] - q_min) / (q_max - q_min)
    if q_max > q_min else 0.0
)

WEIGHT_RUL = 0.75
WEIGHT_RISK = 0.20
WEIGHT_UNCERTAINTY = 0.05

best_predictions["priority_score"] = (
    WEIGHT_RUL * best_predictions["rul_urgency_normalized"]
    + WEIGHT_RISK * best_predictions["risk_normalized"]
    + WEIGHT_UNCERTAINTY * best_predictions["uncertainty_normalized"]
)

display(best_predictions.head())

In [ ]:
class MaintenanceDecisionAgent:
    """Capacity-constrained utility-based maintenance decision-support agent."""

    VALID_POLICIES = {"first_observed", "predicted_rul", "utility"}

    def __init__(self, capacity, policy="utility"):
        if capacity <= 0:
            raise ValueError("Capacity must be greater than zero.")
        if policy not in self.VALID_POLICIES:
            raise ValueError(f"Policy must be one of {self.VALID_POLICIES}.")
        self.capacity = int(capacity)
        self.policy = policy

    def act(self, engine_data):
        ranked = engine_data.copy()

        if self.policy == "first_observed":
            ranked = ranked.sort_values(["unit_id"])
        elif self.policy == "predicted_rul":
            ranked = ranked.sort_values(
                ["predicted_rul", "unit_id"], ascending=[True, True]
            )
        else:
            ranked = ranked.sort_values(
                ["priority_score", "predicted_rul", "unit_id"],
                ascending=[False, True, True]
            )

        ranked = ranked.reset_index(drop=True)
        ranked["queue_position"] = np.arange(1, len(ranked) + 1)
        ranked["selected_for_maintenance"] = (
            ranked["queue_position"] <= self.capacity
        )
        ranked["recommended_action"] = np.where(
            ranked["selected_for_maintenance"], "Service", "Defer"
        )
        ranked["policy"] = self.policy
        ranked["capacity"] = self.capacity
        return ranked

In [ ]:
def create_ideal_queue(engine_data, capacity):
    ideal = (
        engine_data.sort_values(["actual_rul", "unit_id"])
        .reset_index(drop=True)
        .copy()
    )
    ideal["queue_position"] = np.arange(1, len(ideal) + 1)
    ideal["selected_for_maintenance"] = ideal["queue_position"] <= capacity
    return ideal

def critical_recall_at_k(queue):
    actual_critical = queue["actual_rul"] <= CRITICAL_THRESHOLD
    selected = queue["selected_for_maintenance"]
    denominator = actual_critical.sum()
    return float((actual_critical & selected).sum() / denominator) if denominator else np.nan

def missed_critical_engines(queue):
    return int((
        (queue["actual_rul"] <= CRITICAL_THRESHOLD)
        & (~queue["selected_for_maintenance"])
    ).sum())

def queue_regret(queue, ideal_queue):
    selected_sum = queue.loc[
        queue["selected_for_maintenance"], "actual_rul"
    ].sum()
    ideal_sum = ideal_queue.loc[
        ideal_queue["selected_for_maintenance"], "actual_rul"
    ].sum()
    return float(selected_sum - ideal_sum)

def ideal_queue_overlap(queue, ideal_queue):
    selected_ids = set(queue.loc[
        queue["selected_for_maintenance"], "unit_id"
    ])
    ideal_ids = set(ideal_queue.loc[
        ideal_queue["selected_for_maintenance"], "unit_id"
    ])
    return float(len(selected_ids & ideal_ids) / len(ideal_ids)) if ideal_ids else np.nan

def deferred_failure_penalty(queue):
    risky_deferred = queue[
        (~queue["selected_for_maintenance"])
        & (queue["actual_rul"] <= CRITICAL_THRESHOLD)
    ]
    if risky_deferred.empty:
        return 0.0
    penalties = np.exp(
        (CRITICAL_THRESHOLD - risky_deferred["actual_rul"]) / 10
    ) - 1
    return float(penalties.sum())

def evaluate_policy(queue, ideal_queue, policy_name):
    selected = queue[queue["selected_for_maintenance"]]
    return {
        "policy": policy_name,
        "capacity": int(queue["capacity"].iloc[0]),
        "critical_recall_at_k": critical_recall_at_k(queue),
        "missed_critical_engines": missed_critical_engines(queue),
        "average_actual_rul_selected": float(selected["actual_rul"].mean()),
        "queue_regret": queue_regret(queue, ideal_queue),
        "ideal_queue_overlap": ideal_queue_overlap(queue, ideal_queue),
        "deferred_failure_penalty": deferred_failure_penalty(queue)
    }

In [ ]:
MAINTENANCE_CAPACITY = 5

policy_labels = {
    "first_observed": "First Observed",
    "predicted_rul": "Predicted RUL",
    "utility": "Utility-Based Agent"
}

ideal_queue = create_ideal_queue(best_predictions, MAINTENANCE_CAPACITY)
queues = {}
results = []

for policy_key, policy_label in policy_labels.items():
    agent = MaintenanceDecisionAgent(
        capacity=MAINTENANCE_CAPACITY,
        policy=policy_key
    )
    queue = agent.act(best_predictions)
    queues[policy_label] = queue
    results.append(evaluate_policy(queue, ideal_queue, policy_label))

policy_comparison = pd.DataFrame(results)
display(policy_comparison)

utility_queue = queues["Utility-Based Agent"]
final_maintenance_queue = utility_queue[
    [
        "queue_position", "unit_id", "predicted_rul", "actual_rul",
        "predicted_risk_tier", "actual_risk_tier", "priority_score",
        "recommended_action"
    ]
].copy()

final_maintenance_queue["predicted_rul"] = (
    final_maintenance_queue["predicted_rul"].round(2)
)
final_maintenance_queue["priority_score"] = (
    final_maintenance_queue["priority_score"].round(4)
)

display(final_maintenance_queue.head(15))

final_maintenance_queue.to_csv(
    FINAL_OUTPUT_DIR / "final_maintenance_queue.csv", index=False
)
policy_comparison.to_csv(
    FINAL_OUTPUT_DIR / "maintenance_policy_comparison.csv", index=False
)

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(policy_comparison["policy"], policy_comparison["queue_regret"])
plt.title(f"Queue Regret at Capacity {MAINTENANCE_CAPACITY}")
plt.xlabel("Policy")
plt.ylabel("Queue Regret — Lower Is Better")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.bar(
    policy_comparison["policy"],
    policy_comparison["critical_recall_at_k"]
)
plt.title(f"Critical-Engine Recall at Capacity {MAINTENANCE_CAPACITY}")
plt.xlabel("Policy")
plt.ylabel("Critical Recall")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Capacity-Sensitivity Analysis

In [ ]:
CAPACITY_LEVELS = [3, 5, 10, 15, 20]
capacity_rows = []

for capacity in CAPACITY_LEVELS:
    current_ideal = create_ideal_queue(best_predictions, capacity)

    for policy_key, policy_label in policy_labels.items():
        queue = MaintenanceDecisionAgent(
            capacity=capacity,
            policy=policy_key
        ).act(best_predictions)

        capacity_rows.append(
            evaluate_policy(queue, current_ideal, policy_label)
        )

capacity_comparison = pd.DataFrame(capacity_rows)
display(capacity_comparison.head(10))

capacity_comparison.to_csv(
    FINAL_OUTPUT_DIR / "capacity_sensitivity_analysis.csv",
    index=False
)

plt.figure(figsize=(9, 5))
for policy_name in capacity_comparison["policy"].unique():
    policy_data = capacity_comparison[
        capacity_comparison["policy"] == policy_name
    ]
    plt.plot(
        policy_data["capacity"],
        policy_data["critical_recall_at_k"],
        marker="o",
        label=policy_name
    )

plt.title("Critical-Engine Recall by Maintenance Capacity")
plt.xlabel("Available Maintenance Slots")
plt.ylabel("Critical Recall")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()

## Operationally Dangerous Prediction Errors

In [ ]:
dangerous_overestimates = (
    best_predictions[best_predictions["error"] > 0]
    .sort_values("error", ascending=False)
)

missed_risk_cases = best_predictions[
    (best_predictions["actual_risk_tier"] == "Critical")
    & (best_predictions["predicted_risk_tier"] != "Critical")
]

print("Largest RUL overestimates:")
display(dangerous_overestimates[
    [
        "unit_id", "actual_rul", "predicted_rul", "error",
        "actual_risk_tier", "predicted_risk_tier"
    ]
].head(10))

print("Actually critical engines not predicted as critical:")
display(missed_risk_cases[
    [
        "unit_id", "actual_rul", "predicted_rul", "error",
        "actual_risk_tier", "predicted_risk_tier"
    ]
])

# Part 7 — Final Findings and Business Interpretation

Complete this section after running the notebook.

## Predictive-model findings
- Best model: **___**
- Test RMSE: **___**
- Test MAE: **___**
- C-MAPSS score: **___**
- Main predictive drivers: **___**

## Maintenance-agent findings
- Critical recall at capacity five: **___**
- Missed critical engines: **___**
- Queue regret: **___**
- Best policy: **___**
- Effect of increasing capacity: **___**

## Limitations
- FD001 is simulated and contains one operating condition and one fault mode.
- Risk thresholds, capacity, and utility weights are project assumptions.
- Risk tier is derived from RUL rather than independently observed.
- The system recommends maintenance decisions but does not execute them.
- Literature results may use different preprocessing and validation designs.

## Future work
- Evaluate FD002–FD004.
- Add calibrated uncertainty estimates.
- Incorporate maintenance cost, service duration, bay compatibility, and technician availability.
- Compare greedy policies with formal optimization when richer constraints are available.